In [ ]:
!pip install shap lime scikit-learn pandas matplotlib seaborn --quiet
print("✅ All libraries installed successfully!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import lime
import lime.lime_tabular
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
shap.initjs()
print("✅ Imports complete!")

In [ ]:
# ── Upload breast_cancer.csv manually before running this cell ──
# Place the CSV in the same folder as this notebook

df = pd.read_csv('breast_cancer.csv')

feature_names = [c for c in df.columns if c != 'target']
class_names   = ['malignant', 'benign']

X = df[feature_names].values
y = df['target'].values

print(f'Dataset shape     : {df.shape}')
print(f'Features          : {len(feature_names)}')
print(f'Malignant (0)     : {(y==0).sum()} patients')
print(f'Benign    (1)     : {(y==1).sum()} patients')
df.head()

In [ ]:
features_to_plot = [
    'mean radius', 'mean texture', 'mean perimeter', 'mean area',
    'mean smoothness', 'mean concavity', 'mean concave points', 'mean symmetry'
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Feature Distributions by Diagnosis (Malignant vs Benign)', fontsize=14, fontweight='bold')

for ax, feat in zip(axes.flatten(), features_to_plot):
    for target_val, color, name in [(0,'#E24B4A','Malignant'), (1,'#5DCAA5','Benign')]:
        subset = df[df['target'] == target_val][feat]
        ax.hist(subset, bins=20, alpha=0.6, color=color, label=name, edgecolor='white')
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel(feat)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Notice which features separate malignant from benign most clearly — SHAP will confirm this!')

In [ ]:
# Top 10 features by correlation with target
corr = df.corr()['target'].drop('target').abs().sort_values(ascending=False)
top10 = corr.head(10).index.tolist()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    df[top10 + ['target']].corr(),
    annot=True, fmt='.2f', cmap='RdYlGn', center=0,
    linewidths=0.5, ax=ax
)
ax.set_title('Correlation Heatmap — Top 10 Features vs Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top correlated features:')
print(corr.head(10).to_string())

In [ ]:
df = pd.read_csv('breast_cancer.csv')
feature_names = [c for c in df.columns if c != 'target']
class_names   = ['malignant', 'benign']
X = df[feature_names].values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# -------------------------------------------------------
# STUDENT TASK: Try changing n_estimators (50, 100, 200)
#               and max_depth (None, 5, 10)
# -------------------------------------------------------
model = RandomForestClassifier(
    n_estimators=100,   # <-- change me
    max_depth=None,     # <-- change me
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')
print(f'\n✅ Model Accuracy : {acc:.4f} ({acc*100:.1f}%)')
print('\nDetailed Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Malignant','Benign']))

In [ ]:
# ── Auto-setup: runs if you skipped earlier cells ──────────────────
import numpy as np, pandas as pd, warnings, shap, lime, lime.lime_tabular
import matplotlib.patches as mpatches, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings('ignore')
np.random.seed(42)
shap.initjs()

if 'model' not in dir() or 'X_test' not in dir():
    print('Auto-loading data and training model...')
    df = pd.read_csv('breast_cancer.csv')
    feature_names = [c for c in df.columns if c != 'target']
    class_names   = ['malignant', 'benign']
    X = df[feature_names].values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print(f'  Model ready. Accuracy: {model.score(X_test, y_test):.3f}')
else:
    print('Model and data already loaded.')

if 'shap_values' not in dir() or 'explainer' not in dir():
    print('Computing SHAP values...')
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print('  SHAP values ready.')
else:
    print('SHAP values already computed.')

if 'lime_explainer' not in dir():
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train, feature_names=feature_names,
        class_names=['Malignant','Benign'], mode='classification', random_state=42)
    print('LIME explainer ready.')

# shap_values shape: (samples, features, classes)
# We explain class 1 (Benign); class 0 is the mirror image
print(f'SHAP values shape: {shap_values.shape}')
print(f'= ({len(X_test)} patients, {len(feature_names)} features, 2 classes)')

plt.figure(figsize=(11, 8))
shap.summary_plot(
    shap_values[:, :, 1],          # class 1 = Benign
    X_test,
    feature_names=feature_names,
    plot_type='dot',
    show=False
)
plt.title('SHAP Summary Plot — Global Feature Importance (Benign class)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 X-axis = SHAP value. Color = feature value (red=high, blue=low).')
print('💡 Features ranked by mean |SHAP| — most important at the top.')

In [ ]:
# ── Auto-setup: runs if you skipped earlier cells ──────────────────
import numpy as np, pandas as pd, warnings, shap, lime, lime.lime_tabular
import matplotlib.patches as mpatches, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings('ignore')
np.random.seed(42)
shap.initjs()

if 'model' not in dir() or 'X_test' not in dir():
    print('Auto-loading data and training model...')
    df = pd.read_csv('breast_cancer.csv')
    feature_names = [c for c in df.columns if c != 'target']
    class_names   = ['malignant', 'benign']
    X = df[feature_names].values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print(f'  Model ready. Accuracy: {model.score(X_test, y_test):.3f}')
else:
    print('Model and data already loaded.')

if 'shap_values' not in dir() or 'explainer' not in dir():
    print('Computing SHAP values...')
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print('  SHAP values ready.')
else:
    print('SHAP values already computed.')

if 'lime_explainer' not in dir():
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train, feature_names=feature_names,
        class_names=['Malignant','Benign'], mode='classification', random_state=42)
    print('LIME explainer ready.')

mean_shap = np.abs(shap_values[:, :, 1]).mean(axis=0)
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean |SHAP|': mean_shap
}).sort_values('Mean |SHAP|', ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(importance_df['Feature'], importance_df['Mean |SHAP|'],
               color='#378ADD', edgecolor='white', height=0.6)
ax.set_xlabel('Mean |SHAP value|', fontsize=11)
ax.set_title('Global Feature Importance via SHAP — Breast Cancer', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar, val in zip(bars, importance_df['Mean |SHAP|']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 most important features:')
print(importance_df.tail(5)[['Feature','Mean |SHAP|']].to_string(index=False))

In [ ]:
# ── Auto-setup: runs if you skipped earlier cells ──────────────────
import numpy as np, pandas as pd, warnings, shap, lime, lime.lime_tabular
import matplotlib.patches as mpatches, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings('ignore')
np.random.seed(42)
shap.initjs()

if 'model' not in dir() or 'X_test' not in dir():
    print('Auto-loading data and training model...')
    df = pd.read_csv('breast_cancer.csv')
    feature_names = [c for c in df.columns if c != 'target']
    class_names   = ['malignant', 'benign']
    X = df[feature_names].values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print(f'  Model ready. Accuracy: {model.score(X_test, y_test):.3f}')
else:
    print('Model and data already loaded.')

if 'shap_values' not in dir() or 'explainer' not in dir():
    print('Computing SHAP values...')
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print('  SHAP values ready.')
else:
    print('SHAP values already computed.')

if 'lime_explainer' not in dir():
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train, feature_names=feature_names,
        class_names=['Malignant','Benign'], mode='classification', random_state=42)
    print('LIME explainer ready.')

# -------------------------------------------------------
# STUDENT TASK: Change patient_idx to any value 0 to len(X_test)-1
#               Observe how the explanation changes!
# -------------------------------------------------------
patient_idx = 0   # <-- change me

patient    = X_test[patient_idx]
true_label = y_test[patient_idx]
pred_prob  = model.predict_proba(patient.reshape(1,-1))[0]
pred_label = model.predict(patient.reshape(1,-1))[0]

print(f'Patient #{patient_idx}')
print(f'True label  : {class_names[true_label]}')
print(f'Predicted   : {class_names[pred_label]} (prob benign={pred_prob[1]:.3f})')
print()
print('Patient feature values (first 10):')
for name, val in zip(feature_names[:10], patient[:10]):
    print(f'  {name:30s}: {val:.4f}')

# Waterfall bar chart
patient_shap = shap_values[patient_idx, :, 1]
base_value   = explainer.expected_value[1]

shap_df = pd.DataFrame({
    'Feature': [f"{name}={val:.2f}" for name, val in zip(feature_names, patient)],
    'SHAP': patient_shap
}).sort_values('SHAP', key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#E24B4A' if v > 0 else '#1D9E75' for v in shap_df['SHAP']]
ax.barh(shap_df['Feature'], shap_df['SHAP'], color=colors, edgecolor='white', height=0.6)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('SHAP value (impact on Benign prediction)', fontsize=11)
ax.set_title(
    f'SHAP Waterfall — Patient #{patient_idx}\n'
    f'Prediction: {class_names[pred_label]} (prob benign={pred_prob[1]:.2f})',
    fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
red_patch   = mpatches.Patch(color='#E24B4A', label='Pushes toward Benign')
green_patch = mpatches.Patch(color='#1D9E75', label='Pushes toward Malignant')
ax.legend(handles=[red_patch, green_patch], fontsize=9)
plt.tight_layout()
plt.savefig(f'shap_waterfall_patient{patient_idx}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Auto-setup: runs if you skipped earlier cells ──────────────────
import numpy as np, pandas as pd, warnings, shap, lime, lime.lime_tabular
import matplotlib.patches as mpatches, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings('ignore')
np.random.seed(42)
shap.initjs()

if 'model' not in dir() or 'X_test' not in dir():
    print('Auto-loading data and training model...')
    df = pd.read_csv('breast_cancer.csv')
    feature_names = [c for c in df.columns if c != 'target']
    class_names   = ['malignant', 'benign']
    X = df[feature_names].values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print(f'  Model ready. Accuracy: {model.score(X_test, y_test):.3f}')
else:
    print('Model and data already loaded.')

if 'shap_values' not in dir() or 'explainer' not in dir():
    print('Computing SHAP values...')
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print('  SHAP values ready.')
else:
    print('SHAP values already computed.')

if 'lime_explainer' not in dir():
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train, feature_names=feature_names,
        class_names=['Malignant','Benign'], mode='classification', random_state=42)
    print('LIME explainer ready.')

force_plot = shap.force_plot(
    base_value=explainer.expected_value[1],
    shap_values=shap_values[patient_idx, :, 1],
    features=X_test[patient_idx],
    feature_names=feature_names
)
shap.save_html('force_plot.html', force_plot)
force_plot

In [ ]:
# ── Auto-setup: runs if you skipped earlier cells ──────────────────
import numpy as np, pandas as pd, warnings, shap, lime, lime.lime_tabular
import matplotlib.patches as mpatches, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings('ignore')
np.random.seed(42)
shap.initjs()

if 'model' not in dir() or 'X_test' not in dir():
    print('Auto-loading data and training model...')
    df = pd.read_csv('breast_cancer.csv')
    feature_names = [c for c in df.columns if c != 'target']
    class_names   = ['malignant', 'benign']
    X = df[feature_names].values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print(f'  Model ready. Accuracy: {model.score(X_test, y_test):.3f}')
else:
    print('Model and data already loaded.')

if 'shap_values' not in dir() or 'explainer' not in dir():
    print('Computing SHAP values...')
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print('  SHAP values ready.')
else:
    print('SHAP values already computed.')

if 'lime_explainer' not in dir():
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train, feature_names=feature_names,
        class_names=['Malignant','Benign'], mode='classification', random_state=42)
    print('LIME explainer ready.')

# -------------------------------------------------------
# STUDENT TASK: Change the feature below.
# Try: 'worst radius', 'worst concave points', 'mean concavity',
#      'worst perimeter', 'mean radius', 'worst area'
# -------------------------------------------------------
feature_to_plot = 'worst radius'   # <-- change me

fig, ax = plt.subplots(figsize=(9, 5))
shap.dependence_plot(
    feature_to_plot,
    shap_values[:, :, 1],
    X_test,
    feature_names=feature_names,
    ax=ax,
    show=False
)
ax.set_title(f'SHAP Dependence Plot: {feature_to_plot}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'shap_dependence_{feature_to_plot.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 X-axis = feature value. Y-axis = SHAP value. Color = interacting feature.')

In [ ]:
# ── Auto-setup: runs if you skipped earlier cells ──────────────────
import numpy as np, pandas as pd, warnings, shap, lime, lime.lime_tabular
import matplotlib.patches as mpatches, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings('ignore')
np.random.seed(42)
shap.initjs()

if 'model' not in dir() or 'X_test' not in dir():
    print('Auto-loading data and training model...')
    df = pd.read_csv('breast_cancer.csv')
    feature_names = [c for c in df.columns if c != 'target']
    class_names   = ['malignant', 'benign']
    X = df[feature_names].values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print(f'  Model ready. Accuracy: {model.score(X_test, y_test):.3f}')
else:
    print('Model and data already loaded.')

if 'shap_values' not in dir() or 'explainer' not in dir():
    print('Computing SHAP values...')
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print('  SHAP values ready.')
else:
    print('SHAP values already computed.')

if 'lime_explainer' not in dir():
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train, feature_names=feature_names,
        class_names=['Malignant','Benign'], mode='classification', random_state=42)
    print('LIME explainer ready.')

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train,
    feature_names=feature_names,
    class_names=['Malignant', 'Benign'],
    mode='classification',
    random_state=42
)

print('✅ LIME explainer created!')
print(f'Training distribution shape: {X_train.shape}')
print('The explainer will sample perturbations from this training distribution.')

In [ ]:
# ── Auto-setup: runs if you skipped earlier cells ──────────────────
import numpy as np, pandas as pd, warnings, shap, lime, lime.lime_tabular
import matplotlib.patches as mpatches, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings('ignore')
np.random.seed(42)
shap.initjs()

if 'model' not in dir() or 'X_test' not in dir():
    print('Auto-loading data and training model...')
    df = pd.read_csv('breast_cancer.csv')
    feature_names = [c for c in df.columns if c != 'target']
    class_names   = ['malignant', 'benign']
    X = df[feature_names].values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print(f'  Model ready. Accuracy: {model.score(X_test, y_test):.3f}')
else:
    print('Model and data already loaded.')

if 'shap_values' not in dir() or 'explainer' not in dir():
    print('Computing SHAP values...')
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print('  SHAP values ready.')
else:
    print('SHAP values already computed.')

if 'lime_explainer' not in dir():
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train, feature_names=feature_names,
        class_names=['Malignant','Benign'], mode='classification', random_state=42)
    print('LIME explainer ready.')

# -------------------------------------------------------
# STUDENT TASK: Change patient_idx and num_features
#               Observe how the explanation changes!
# -------------------------------------------------------
patient_idx  = 0      # <-- change me (0 to len(X_test)-1)
num_features = 10     # <-- change me (try 5, 10, or 30)

lime_exp = lime_explainer.explain_instance(
    data_row=X_test[patient_idx],
    predict_fn=model.predict_proba,
    num_features=num_features,
    num_samples=1000
)

pred_prob_lime = model.predict_proba(X_test[patient_idx].reshape(1,-1))[0]
print(f'Patient #{patient_idx} — LIME Explanation')
print(f'Predicted prob of Benign  : {pred_prob_lime[1]:.3f}')
print(f'Predicted prob of Malignant: {pred_prob_lime[0]:.3f}')
print()
print('Feature conditions and their weights (for Benign class):')
for feat, weight in lime_exp.as_list():
    direction = '↑ supports benign' if weight > 0 else '↓ supports malignant'
    print(f'  {feat:40s}: {weight:+.4f}  ({direction})')

In [ ]:
# ── Auto-setup: runs if you skipped earlier cells ──────────────────
import numpy as np, pandas as pd, warnings, shap, lime, lime.lime_tabular
import matplotlib.patches as mpatches, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings('ignore')
np.random.seed(42)
shap.initjs()

if 'model' not in dir() or 'X_test' not in dir():
    print('Auto-loading data and training model...')
    df = pd.read_csv('breast_cancer.csv')
    feature_names = [c for c in df.columns if c != 'target']
    class_names   = ['malignant', 'benign']
    X = df[feature_names].values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print(f'  Model ready. Accuracy: {model.score(X_test, y_test):.3f}')
else:
    print('Model and data already loaded.')

if 'shap_values' not in dir() or 'explainer' not in dir():
    print('Computing SHAP values...')
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print('  SHAP values ready.')
else:
    print('SHAP values already computed.')

if 'lime_explainer' not in dir():
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train, feature_names=feature_names,
        class_names=['Malignant','Benign'], mode='classification', random_state=42)
    print('LIME explainer ready.')

lime_list     = lime_exp.as_list()
features_lime = [x[0] for x in lime_list]
weights_lime  = [x[1] for x in lime_list]

colors_lime = ['#E24B4A' if w > 0 else '#1D9E75' for w in weights_lime]

fig, ax = plt.subplots(figsize=(11, 7))
y_pos = range(len(features_lime))
ax.barh(y_pos, weights_lime, color=colors_lime, edgecolor='white', height=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(features_lime, fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('LIME weight (local linear model coefficient)', fontsize=11)
ax.set_title(f'LIME Explanation — Patient #{patient_idx}\nTop {num_features} features',
             fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
red_patch   = mpatches.Patch(color='#E24B4A', label='Supports Benign prediction')
green_patch = mpatches.Patch(color='#1D9E75', label='Supports Malignant prediction')
ax.legend(handles=[red_patch, green_patch], fontsize=9)
plt.tight_layout()
plt.savefig(f'lime_patient{patient_idx}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Auto-setup: runs if you skipped earlier cells ──────────────────
import numpy as np, pandas as pd, warnings, shap, lime, lime.lime_tabular
import matplotlib.patches as mpatches, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings('ignore')
np.random.seed(42)
shap.initjs()

if 'model' not in dir() or 'X_test' not in dir():
    print('Auto-loading data and training model...')
    df = pd.read_csv('breast_cancer.csv')
    feature_names = [c for c in df.columns if c != 'target']
    class_names   = ['malignant', 'benign']
    X = df[feature_names].values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print(f'  Model ready. Accuracy: {model.score(X_test, y_test):.3f}')
else:
    print('Model and data already loaded.')

if 'shap_values' not in dir() or 'explainer' not in dir():
    print('Computing SHAP values...')
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print('  SHAP values ready.')
else:
    print('SHAP values already computed.')

if 'lime_explainer' not in dir():
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train, feature_names=feature_names,
        class_names=['Malignant','Benign'], mode='classification', random_state=42)
    print('LIME explainer ready.')

# LIME's built-in HTML display
try:
    lime_exp.show_in_notebook(show_table=True, show_all=False)
except Exception:
    from IPython.display import display, HTML
    display(HTML(lime_exp.as_html()))

In [ ]:
# ── Auto-setup: runs if you skipped earlier cells ──────────────────
import numpy as np, pandas as pd, warnings, shap, lime, lime.lime_tabular
import matplotlib.patches as mpatches, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings('ignore')
np.random.seed(42)
shap.initjs()

if 'model' not in dir() or 'X_test' not in dir():
    print('Auto-loading data and training model...')
    df = pd.read_csv('breast_cancer.csv')
    feature_names = [c for c in df.columns if c != 'target']
    class_names   = ['malignant', 'benign']
    X = df[feature_names].values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print(f'  Model ready. Accuracy: {model.score(X_test, y_test):.3f}')
else:
    print('Model and data already loaded.')

if 'shap_values' not in dir() or 'explainer' not in dir():
    print('Computing SHAP values...')
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print('  SHAP values ready.')
else:
    print('SHAP values already computed.')

if 'lime_explainer' not in dir():
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train, feature_names=feature_names,
        class_names=['Malignant','Benign'], mode='classification', random_state=42)
    print('LIME explainer ready.')

# -------------------------------------------------------
# STUDENT TASK: Try different patient indices.
#               Do SHAP and LIME always agree?
# -------------------------------------------------------
patient_idx = 0   # <-- change me

# SHAP values for this patient (class 1 = Benign)
shap_patient = shap_values[patient_idx, :, 1]
shap_df_compare = pd.DataFrame({'Feature': feature_names, 'SHAP': shap_patient})
shap_df_compare['abs_SHAP'] = shap_df_compare['SHAP'].abs()
shap_df_compare = shap_df_compare.sort_values('abs_SHAP', ascending=False).head(8)

# LIME values for same patient
lime_exp2 = lime_explainer.explain_instance(
    X_test[patient_idx], model.predict_proba, num_features=8, num_samples=1000)

# Plot side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'SHAP vs LIME — Patient #{patient_idx} | '
             f'True: {class_names[y_test[patient_idx]]} | '
             f'Pred: {class_names[model.predict(X_test[[patient_idx]])[0]]}',
             fontsize=13, fontweight='bold')

# SHAP panel
shap_sorted = shap_df_compare.sort_values('SHAP')
colors_s = ['#E24B4A' if v > 0 else '#1D9E75' for v in shap_sorted['SHAP']]
ax1.barh(shap_sorted['Feature'], shap_sorted['SHAP'], color=colors_s, edgecolor='white', height=0.6)
ax1.axvline(0, color='black', linewidth=0.8)
ax1.set_title('SHAP Values\n(game-theory, consistent)', fontsize=11)
ax1.set_xlabel('SHAP value')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# LIME panel
lime_items = lime_exp2.as_list()
lime_feats = [x[0] for x in lime_items]
lime_vals  = [x[1] for x in lime_items]
colors_l = ['#E24B4A' if v > 0 else '#1D9E75' for v in lime_vals]
ax2.barh(lime_feats, lime_vals, color=colors_l, edgecolor='white', height=0.6)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title('LIME Weights\n(local surrogate, approximate)', fontsize=11)
ax2.set_xlabel('LIME weight')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'shap_vs_lime_patient{patient_idx}.png', dpi=150, bbox_inches='tight')
plt.show()

print('Reflection questions:')
print('  1. Do both methods agree on the TOP feature?')
print('  2. Does LIME show raw values or rule conditions?')
print('  3. Which is easier to explain to a clinician — SHAP or LIME?')

In [ ]:
# ── Auto-setup: runs if you skipped earlier cells ──────────────────
import numpy as np, pandas as pd, warnings, shap, lime, lime.lime_tabular
import matplotlib.patches as mpatches, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings('ignore')
np.random.seed(42)
shap.initjs()

if 'model' not in dir() or 'X_test' not in dir():
    print('Auto-loading data and training model...')
    df = pd.read_csv('breast_cancer.csv')
    feature_names = [c for c in df.columns if c != 'target']
    class_names   = ['malignant', 'benign']
    X = df[feature_names].values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print(f'  Model ready. Accuracy: {model.score(X_test, y_test):.3f}')
else:
    print('Model and data already loaded.')

if 'shap_values' not in dir() or 'explainer' not in dir():
    print('Computing SHAP values...')
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print('  SHAP values ready.')
else:
    print('SHAP values already computed.')

if 'lime_explainer' not in dir():
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train, feature_names=feature_names,
        class_names=['Malignant','Benign'], mode='classification', random_state=42)
    print('LIME explainer ready.')

# Step 1: Find the most uncertain patient
probs       = model.predict_proba(X_test)[:, 1]
uncertainty = np.abs(probs - 0.5)
hardest_idx = int(np.argmin(uncertainty))

print(f'Most uncertain patient index : {hardest_idx}')
print(f'Predicted prob of Benign     : {probs[hardest_idx]:.4f}')
print(f'True label                   : {class_names[y_test[hardest_idx]]}')

# -------------------------------------------------------
# STUDENT TASK: Explain this patient using SHAP and LIME.
# Copy the code from Steps 8 and 11,
# replace patient_idx with hardest_idx.
# Write your analysis in a markdown cell below.
# -------------------------------------------------------

# YOUR CODE HERE
# shap explanation for hardest_idx ...
# lime explanation for hardest_idx ...

In [ ]:
# ── Auto-setup: runs if you skipped earlier cells ──────────────────
import numpy as np, pandas as pd, warnings, shap, lime, lime.lime_tabular
import matplotlib.patches as mpatches, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings('ignore')
np.random.seed(42)
shap.initjs()

if 'model' not in dir() or 'X_test' not in dir():
    print('Auto-loading data and training model...')
    df = pd.read_csv('breast_cancer.csv')
    feature_names = [c for c in df.columns if c != 'target']
    class_names   = ['malignant', 'benign']
    X = df[feature_names].values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print(f'  Model ready. Accuracy: {model.score(X_test, y_test):.3f}')
else:
    print('Model and data already loaded.')

if 'shap_values' not in dir() or 'explainer' not in dir():
    print('Computing SHAP values...')
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print('  SHAP values ready.')
else:
    print('SHAP values already computed.')

if 'lime_explainer' not in dir():
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train, feature_names=feature_names,
        class_names=['Malignant','Benign'], mode='classification', random_state=42)
    print('LIME explainer ready.')

# Show SHAP decision plot for first 20 test patients
plt.figure(figsize=(10, 8))
shap.decision_plot(
    base_value=explainer.expected_value[1],
    shap_values=shap_values[:20, :, 1],
    features=X_test[:20],
    feature_names=feature_names,
    show=False
)
plt.title('SHAP Decision Plot — First 20 Test Patients', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_decision_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Each line = one patient. Watch how features push predictions left (malignant) or right (benign).')